In [10]:
 %run ../setup/config

Box(children=(Label(value='Environment'), Dropdown(options=('dev', 'prod'), value='dev')))

Box(children=(Label(value='Cluster ID (vacío = serverless)'), Text(value='')))

In [ ]:
import base64
from pyspark.sql.functions import col, current_timestamp

In [ ]:
dbutils.widgets.dropdown(
    "trigger_type", "availableNow", ["availableNow", "once", "processingTime"]
)
trigger_type = dbutils.widgets.get("trigger_type")

Box(children=(Label(value='trigger_type'), Dropdown(options=('availableNow', 'once', 'processingTime'), value=…

In [ ]:
api_key_b64 = dbutils.secrets.get(scope="confluent-scope", key="api-key")
api_secret_b64 = dbutils.secrets.get(scope="confluent-scope", key="api-secret")

api_key = base64.b64decode(api_key_b64).decode("utf-8")
api_secret = base64.b64decode(api_secret_b64).decode("utf-8")

In [ ]:
kafka_options = {
    "kafka.bootstrap.servers": BOOTSTRAP_SERVERS,
    "subscribe": TOPIC_NAME,
    "kafka.security.protocol": "SASL_SSL",
    "kafka.sasl.mechanism": "PLAIN",
    "kafka.sasl.jaas.config": (
        "kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required "
        f'username="{api_key}" password="{api_secret}";'
    ),
    "startingOffsets": "earliest",
}

In [ ]:
df_raw = spark.readStream.format("kafka").options(**kafka_options).load()

df_bronze = (
    df_raw
    .withColumn("raw_json", col("value").cast("string"))
    .withColumn("_ingestion_timestamp", current_timestamp())
    .select("raw_json", "topic", "partition", "offset", "timestamp", "_ingestion_timestamp")
)

trigger_options = {
    "availableNow": {"availableNow": True},
    "once": {"once": True},
    "processingTime": {"processingTime": "10 seconds"},
}

query = (
    df_bronze.writeStream
    .format("delta")
    .option("checkpointLocation", ORDERS_CHECKPOINT_LOCATION)
    .trigger(**trigger_options[trigger_type])
    .toTable(BRONZE_ORDER_FULL_TABLE)
)


if trigger_type in ("availableNow", "once"):
    query.awaitTermination()

Ingestion complete into dbr_dev.bronze.brz_orders_events
